In [ ]:
# ============================================================
# RF_PI Hyperparameter Grid Search
# Mean and standard deviation of performance across 50 seeds
# Station-level 70:30 training-validation split
#
# Author: Junyoung Lee
# Affiliation: Ulsan National Institute of Science and Technology (UNIST)
# Email: junyounglee@unist.ac.kr
# ============================================================

from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


# ------------------------------------------------------------
# Repository paths
# ------------------------------------------------------------
# This notebook can be run from either the repository root
# or the notebooks directory.
current_dir = Path.cwd()
project_dir = (
    current_dir.parent
    if current_dir.name == 'notebooks'
    else current_dir
)

data_dir = project_dir / 'data'
output_dir = project_dir / 'results'
output_dir.mkdir(parents=True, exist_ok=True)

input_file = data_dir / 'Total data_for submission.csv'
total_data = pd.read_csv(input_file)

summary_output_file = output_dir / '01-RF_PI_hyperparameter_summary.csv'

# Use the absolute value of the U-component ratio
total_data['U.ratio'] = np.abs(total_data['U.ratio'])


# ------------------------------------------------------------
# Analysis settings
# ------------------------------------------------------------
station_col = 'SSN'
seeds = range(1, 51)
training_fraction = 0.70


# ------------------------------------------------------------
# Feature sets
# ------------------------------------------------------------
feature_sets = {
    'Set1': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle'],
    'Set2': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle', 'S.Lat', 'S.Long'],
    'Set3': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle', 'slope_500m'],
    'Set4': ['U.ratio', 'Mw', 'EpiD', 'E.Dep', 'ray.p', 'j.angle', 'S.Lat', 'S.Long', 'slope_500m']
}


# ------------------------------------------------------------
# Hyperparameter grid
# ------------------------------------------------------------
n_estimators_list = [500]
max_features_list = [2, 4, None]
max_depth_list = [2, 3, 5, 7, 9]


# ------------------------------------------------------------
# Prepare the physics-informed residual target
# ------------------------------------------------------------
total_data = total_data.loc[
    (total_data['Vs30_mea'] > 0) &
    (total_data['Vs30_f0'] > 0)
].copy()

total_data['log_res'] = np.log(
    total_data['Vs30_mea'] / total_data['Vs30_f0']
)


# ------------------------------------------------------------
# Retain common complete cases for all feature sets
# ------------------------------------------------------------
all_features = sorted({
    feature
    for features in feature_sets.values()
    for feature in features
})

required_cols = [
    station_col,
    'Vs30_mea',
    'Vs30_f0',
    'log_res',
    *all_features
]

missing_cols = [
    column
    for column in required_cols
    if column not in total_data.columns
]

if missing_cols:
    raise KeyError(
        f'Missing required columns in the input data: {missing_cols}'
    )

total_data = (
    total_data[required_cols]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .copy()
)

stations = np.array(sorted(total_data[station_col].unique()))

if len(stations) < 2:
    raise ValueError(
        'At least two unique stations are required for station-level splitting.'
    )


# ------------------------------------------------------------
# Hyperparameter grid search
# ------------------------------------------------------------
summary_rows = []

parameter_combinations = list(
    product(
        n_estimators_list,
        max_features_list,
        max_depth_list
    )
)

total_combinations = len(parameter_combinations)

for combination_index, (
    n_estimators,
    max_features,
    max_depth
) in enumerate(parameter_combinations, start=1):

    max_features_label = (
        'all' if max_features == 1.0 else str(max_features)
    )

    print(
        f'Running combination '
        f'{combination_index}/{total_combinations}: '
        f'n_estimators={n_estimators}, '
        f'max_features={max_features_label}, '
        f'max_depth={max_depth}'
    )

    seed_results = []

    for seed in seeds:

        # Generate a reproducible station-level split
        rng = np.random.default_rng(seed)
        shuffled_stations = rng.permutation(stations)

        n_training_stations = int(
            len(shuffled_stations) * training_fraction
        )

        training_stations = shuffled_stations[:n_training_stations]
        validation_stations = shuffled_stations[n_training_stations:]

        training_data = total_data.loc[
            total_data[station_col].isin(training_stations)
        ].copy()

        validation_data = total_data.loc[
            total_data[station_col].isin(validation_stations)
        ].copy()

        y_true = validation_data['Vs30_mea'].to_numpy()
        y_pred_pwave = validation_data['Vs30_f0'].to_numpy()

        result_row = {
            'seed': seed,
            'n_training_stations': len(training_stations),
            'n_validation_stations': len(validation_stations),
            'n_training_records': len(training_data),
            'n_validation_records': len(validation_data),
            'RMSE_P-wave': np.sqrt(
                mean_squared_error(y_true, y_pred_pwave)
            ),
            'R2_P-wave': r2_score(y_true, y_pred_pwave)
        }

        for set_name, features in feature_sets.items():

            X_train = training_data[features]
            y_train = training_data['log_res']

            X_validation = validation_data[features]

            model = RandomForestRegressor(
                n_estimators=n_estimators,
                max_features=max_features,
                max_depth=max_depth,
                min_samples_split=5,
                min_samples_leaf=4,
                bootstrap=True,
                random_state=seed,
                n_jobs=-1,
                oob_score=False
            )

            model.fit(X_train, y_train)

            predicted_log_residual = model.predict(X_validation)

            # Physics-informed Vs30 prediction
            y_pred = (
                validation_data['Vs30_f0'].to_numpy() *
                np.exp(predicted_log_residual)
            )

            result_row[f'RMSE_{set_name}'] = np.sqrt(
                mean_squared_error(y_true, y_pred)
            )

            result_row[f'R2_{set_name}'] = r2_score(
                y_true,
                y_pred
            )

        seed_results.append(result_row)

    seed_results_df = pd.DataFrame(seed_results)

    summary_row = {
        'n_estimators': n_estimators,
        'max_features': max_features_label,
        'max_depth': max_depth
    }

    metric_columns = [
        'RMSE_P-wave',
        'RMSE_Set1',
        'RMSE_Set2',
        'RMSE_Set3',
        'RMSE_Set4',
        'R2_P-wave',
        'R2_Set1',
        'R2_Set2',
        'R2_Set3',
        'R2_Set4'
    ]

    for metric in metric_columns:
        summary_row[f'{metric}_mean'] = seed_results_df[metric].mean()
        summary_row[f'{metric}_std'] = seed_results_df[metric].std(ddof=1)

    summary_rows.append(summary_row)


# ------------------------------------------------------------
# Save the grid-search summary
# ------------------------------------------------------------
summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    summary_output_file,
    index=False,
    encoding='utf-8-sig'
)

print(f'\nSaved summary file: {summary_output_file}')
print(summary_df.head())

Running combination 1/15: n_estimators=500, max_features=2, max_depth=2
Running combination 2/15: n_estimators=500, max_features=2, max_depth=3
Running combination 3/15: n_estimators=500, max_features=2, max_depth=5
Running combination 4/15: n_estimators=500, max_features=2, max_depth=7
Running combination 5/15: n_estimators=500, max_features=2, max_depth=9
Running combination 6/15: n_estimators=500, max_features=4, max_depth=2
Running combination 7/15: n_estimators=500, max_features=4, max_depth=3
Running combination 8/15: n_estimators=500, max_features=4, max_depth=5
Running combination 9/15: n_estimators=500, max_features=4, max_depth=7
Running combination 10/15: n_estimators=500, max_features=4, max_depth=9
Running combination 11/15: n_estimators=500, max_features=None, max_depth=2
Running combination 12/15: n_estimators=500, max_features=None, max_depth=3
